
# Metadata → C1 pipeline (clean)

This notebook runs an end-to-end, minimal pipeline:

1. Generate VQA metadata for each frame in a chosen manifest (schema-driven) → JSONL  
2. Run C1 prediction twice on the same frames (WITH metadata vs NO metadata)  
3. Compute metrics (accuracy/confusion + AP + mAP) using binary scores and continuous scores

Only difference between conditions is the metadata block.


In [1]:

from __future__ import annotations

import os, sys, json, re, time, csv
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

from tqdm.auto import tqdm
from PIL import Image

# ---------- REPO / PATHS ----------
REPO_ROOT = Path(os.environ.get("LVLM_REPO_ROOT", "/shared_data0/weiqiuy/lvlm_cvs_reasoning")).resolve()
assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT}"
sys.path.insert(0, str((REPO_ROOT / "src").resolve()))

# Optional API keys (same convention as generate.py / lvlm_test.ipynb)
keys_path = (REPO_ROOT / "API_KEYS2.json").resolve()
if keys_path.exists():
    keys = json.loads(keys_path.read_text())
    os.environ.setdefault("OPENAI_API_KEY", keys.get("OPENAI_API_KEY", ""))
    os.environ.setdefault("ANTHROPIC_API_KEY", keys.get("ANTHROPIC_API_KEY", ""))
    os.environ.setdefault("GOOGLE_API_KEY", keys.get("GOOGLE_API_KEY", ""))

import models  # repo-local

# ---------- CONFIG (edit as needed) ----------
MODEL_ID = "gpt-4.1-mini"

# Video-level manifest (rows contain: id/video_id, split, frame_ids)
MANIFEST_JSONL = (REPO_ROOT / "data" / "train_dev5.jsonl").resolve()

# Frames root: data/frames/<split>/<video_id>/frame_000150.jpg
FRAMES_ROOT = (REPO_ROOT / "data" / "frames").resolve()

# Metadata schema (must include per-question sentences.true/false/unknown)
SCHEMA_PATH = (REPO_ROOT / "schemas" / "tubular_structure_visibility_vqa_v1.json").resolve()

# Outputs
VQA_OUT_JSONL = (REPO_ROOT / "outputs" / "vqa" / f"{SCHEMA_PATH.stem}__{MANIFEST_JSONL.stem}.jsonl").resolve()
OUT_WITH_META = (REPO_ROOT / "outputs" / "c1_pred" / f"c1__with_meta__{MANIFEST_JSONL.stem}.jsonl").resolve()
OUT_NO_META   = (REPO_ROOT / "outputs" / "c1_pred" / f"c1__no_meta__{MANIFEST_JSONL.stem}.jsonl").resolve()

# Ground-truth dataset root:
# data/CVS_Challenge_SAGES_v1/<split>/labels/<video_id>/frame.csv
GT_LABELS_ROOT = (REPO_ROOT / "data" / "CVS_Challenge_SAGES_v1").resolve()
VOTE_RULE = "majority"  # "any" or "majority"

for p in (VQA_OUT_JSONL, OUT_WITH_META, OUT_NO_META):
    p.parent.mkdir(parents=True, exist_ok=True)

assert MANIFEST_JSONL.exists(), f"MANIFEST_JSONL not found: {MANIFEST_JSONL}"
assert FRAMES_ROOT.exists(), f"FRAMES_ROOT not found: {FRAMES_ROOT}"
assert SCHEMA_PATH.exists(), f"SCHEMA_PATH not found: {SCHEMA_PATH}"
assert GT_LABELS_ROOT.exists(), f"GT_LABELS_ROOT not found: {GT_LABELS_ROOT}"

print("REPO_ROOT:", REPO_ROOT)
print("MANIFEST_JSONL:", MANIFEST_JSONL)
print("SCHEMA_PATH:", SCHEMA_PATH)
print("VQA_OUT_JSONL:", VQA_OUT_JSONL)
print("OUT_WITH_META:", OUT_WITH_META)
print("OUT_NO_META:", OUT_NO_META)


REPO_ROOT: /shared_data0/weiqiuy/lvlm_cvs_reasoning
MANIFEST_JSONL: /shared_data0/weiqiuy/lvlm_cvs_reasoning/data/train_dev5.jsonl
SCHEMA_PATH: /shared_data0/weiqiuy/lvlm_cvs_reasoning/schemas/tubular_structure_visibility_vqa_v1.json
VQA_OUT_JSONL: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/vqa/tubular_structure_visibility_vqa_v1__train_dev5.jsonl
OUT_WITH_META: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__with_meta__train_dev5.jsonl
OUT_NO_META: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__no_meta__train_dev5.jsonl


In [2]:

# ---------- MODEL ----------
model = models.create_model(MODEL_ID)
MODEL_ID


'gpt-4.1-mini'

In [3]:

# ---------- JSONL HELPERS ----------
def iter_jsonl(path: Path) -> Iterable[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def append_jsonl(path: Path, row: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def abs_posix(p: Path) -> str:
    return p.resolve().as_posix()

def kf(video_id: str, split: str, frame_id: int) -> Tuple[str, str, int]:
    return (str(video_id), str(split), int(frame_id))


In [4]:

# ---------- FRAMES ----------
def frame_path(split: str, video_id: str, frame_id: int) -> Path:
    return (FRAMES_ROOT / split / video_id / f"frame_{frame_id:06d}.jpg").resolve()

def load_image(p: Path) -> Image.Image:
    return Image.open(p).convert("RGB")


In [5]:

# ---------- LOAD SCHEMA ----------
schema: Dict[str, Any] = json.loads(SCHEMA_PATH.read_text())
schema_task_id = schema.get("task_id", SCHEMA_PATH.stem)

q_by_key: Dict[str, Dict[str, Any]] = {q["key"]: q for q in schema.get("questions", [])}
print("task_id:", schema_task_id)
print("questions:", list(q_by_key.keys()))


task_id: tubular_vqa_v1
questions: ['cystic_duct_visible', 'cystic_artery_visible', 'both_visible', 'duct_artery_separated']


In [6]:

# ---------- VQA PROMPT ----------
def build_vqa_prompt(schema: Dict[str, Any]) -> str:
    qs = schema["questions"]
    lines: List[str] = []
    lines.append("Answer the following questions about the surgical frame.")
    lines.append("Return ONLY valid JSON. No markdown. No extra keys.")
    lines.append('For each key output: {"answer": 0 or 1, "confidence": float in [0,1]}.')
    lines.append("If unsure, choose the most likely answer and lower confidence.")
    lines.append("")
    lines.append("Questions:")
    for q in qs:
        lines.append(f"- {q['key']}: {q['question']}")
    lines.append("")
    tmpl = {q['key']: {"answer": 0, "confidence": 0.0} for q in qs}
    lines.append("JSON TEMPLATE:")
    lines.append(json.dumps(tmpl, indent=2))
    return "\n".join(lines)

SYSTEM_PROMPT_VQA = (
    "You are a careful surgical vision assistant. "
    "Answer only based on what is visible in the provided frame. "
    "Follow the output format exactly."
)

VQA_PROMPT = build_vqa_prompt(schema)


In [7]:

# ---------- JSON PARSING / VALIDATION ----------
def extract_json_obj(text: str) -> Dict[str, Any]:
    t = (text or "").strip()
    if t.startswith("```"):
        lines = t.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        t = "\n".join(lines).strip()

    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found")
    obj = json.loads(m.group(0))
    if not isinstance(obj, dict):
        raise ValueError("JSON is not an object")
    return obj

def validate_vqa(obj: Dict[str, Any], schema: Dict[str, Any]) -> Tuple[bool, List[str]]:
    errs: List[str] = []
    keys = [q["key"] for q in schema["questions"]]
    strict = schema.get("output", {}).get("strict_keys", True)

    if strict:
        extra = set(obj.keys()) - set(keys)
        missing = set(keys) - set(obj.keys())
        if extra:
            errs.append(f"Extra keys: {sorted(extra)}")
        if missing:
            errs.append(f"Missing keys: {sorted(missing)}")

    for k in keys:
        if k not in obj:
            continue
        v = obj[k]
        if not isinstance(v, dict):
            errs.append(f"{k}: must be object")
            continue
        if "answer" not in v:
            errs.append(f"{k}: missing answer")
            continue
        if v["answer"] not in (0, 1):
            errs.append(f"{k}: answer must be 0/1")
        if "confidence" in v:
            try:
                c = float(v["confidence"])
                if not (0.0 <= c <= 1.0):
                    errs.append(f"{k}: confidence out of [0,1]")
            except Exception:
                errs.append(f"{k}: confidence not numeric")
    return (len(errs) == 0), errs


## 1) Generate VQA metadata → `VQA_OUT_JSONL`

In [18]:
REGENERATE_VQA = True
if REGENERATE_VQA and VQA_OUT_JSONL.exists():
    VQA_OUT_JSONL.unlink()
    print("Deleted:", VQA_OUT_JSONL)

# Resume: skip already-written frames
done_vqa = set()
if VQA_OUT_JSONL.exists():
    for r in iter_jsonl(VQA_OUT_JSONL):
        vid = r.get("video_id"); sp = r.get("split"); fid = r.get("frame_id")
        if vid is None or sp is None or fid is None:
            continue

        # only skip if successful OR missing_image
        if r.get("parse_ok", False) or ("missing_image" in (r.get("parse_errors") or [])):
            done_vqa.add(kf(str(vid), str(sp), int(fid)))

manifest_rows = list(iter_jsonl(MANIFEST_JSONL))
print(f"Loaded manifest rows: {len(manifest_rows)}")

n_total = n_written = n_missing = n_parse_fail = 0
t_start = time.time()

for rec in tqdm(manifest_rows, desc="VQA videos"):
    video_id = rec.get("id") or rec.get("video_id")
    split = rec.get("split", "train")
    frame_ids = rec.get("frame_ids") or []
    if video_id is None or not frame_ids:
        continue

    for fid in tqdm(frame_ids):
        try:
            fid_int = int(fid)
        except Exception:
            continue

        n_total += 1
        kk = kf(str(video_id), str(split), fid_int)
        if kk in done_vqa:
            continue

        img_p = frame_path(str(split), str(video_id), fid_int)
        if not img_p.exists():
            n_missing += 1
            append_jsonl(VQA_OUT_JSONL, {
                "task_id": schema_task_id,
                "video_id": str(video_id),
                "split": str(split),
                "frame_id": fid_int,
                "image_path": abs_posix(img_p),
                "qa": {},
                "raw_response": "",
                "parse_ok": False,
                "parse_errors": ["missing_image"],
                "schema_path": abs_posix(SCHEMA_PATH),
                "model": MODEL_ID,
                "vqa_task_id": schema_task_id,
            })
            n_written += 1
            done_vqa.add(kk)
            continue

        img = load_image(img_p)
        t0 = time.time()
        resp = model([(VQA_PROMPT, img)], system_prompt=SYSTEM_PROMPT_VQA)[0]
        latency = time.time() - t0

        try:
            obj = extract_json_obj(resp)
            ok, errs = validate_vqa(obj, schema)
            if not ok:
                n_parse_fail += 1
        except Exception as e:
            obj, ok, errs = {}, False, [f"parse_failed: {type(e).__name__}: {e}"]
            n_parse_fail += 1

        append_jsonl(VQA_OUT_JSONL, {
            "task_id": schema_task_id,
            "video_id": str(video_id),
            "split": str(split),
            "frame_id": fid_int,
            "image_path": abs_posix(img_p),
            "qa": obj,
            "raw_response": resp,
            "parse_ok": ok,
            "parse_errors": errs,
            "schema_path": abs_posix(SCHEMA_PATH),
            "latency_sec": latency,
            "model": MODEL_ID,
            "vqa_task_id": schema_task_id,
        })
        n_written += 1
        done_vqa.add(kk)

elapsed = time.time() - t_start
print(f"VQA done. total={n_total}, written={n_written}, missing_img={n_missing}, parse_fail={n_parse_fail}")
print("Wrote:", VQA_OUT_JSONL)
print(f"Elapsed: {elapsed/60:.1f} min")


Deleted: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/vqa/tubular_structure_visibility_vqa_v1__train_dev5.jsonl
Loaded manifest rows: 5


VQA videos:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/18 [00:00<?, ?it/s]

VQA done. total=90, written=90, missing_img=0, parse_fail=0
Wrote: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/vqa/tubular_structure_visibility_vqa_v1__train_dev5.jsonl
Elapsed: 0.5 min


## 2) Load VQA results + render metadata sentences

In [19]:

def load_vqa_index(path: Path) -> Dict[Tuple[str, str, int], Dict[str, Any]]:
    idx: Dict[Tuple[str, str, int], Dict[str, Any]] = {}
    if not path.exists():
        return idx
    for r in iter_jsonl(path):
        vid = r.get("video_id")
        sp = r.get("split")
        fid = r.get("frame_id")
        if vid is None or sp is None or fid is None:
            continue
        idx[kf(str(vid), str(sp), int(fid))] = r
    return idx

vqa_idx = load_vqa_index(VQA_OUT_JSONL)
print("Loaded VQA rows:", len(vqa_idx))

def render_metadata_sentences(meta_rec: Optional[Dict[str, Any]], schema: Dict[str, Any]) -> Optional[str]:
    if not meta_rec or not meta_rec.get("parse_ok", False):
        return None
    qa = meta_rec.get("qa") or {}

    prefix = schema.get("rendering", {}).get("prefix", "Automated VQA cues (may be noisy):")
    include_conf = bool(schema.get("rendering", {}).get("include_confidence", True))
    conf_fmt = schema.get("rendering", {}).get("confidence_format", " (confidence={conf:.2f})")

    lines: List[str] = [prefix]

    for q in schema.get("questions", []):
        qk = q["key"]
        sents = q.get("sentences", {})
        v = qa.get(qk)

        if not isinstance(v, dict) or "answer" not in v:
            sent = sents.get("unknown") or f"It is unclear whether {qk} is true in this frame."
            lines.append(f"- {sent}")
            continue

        ans = v.get("answer", 0)
        conf = v.get("confidence", 0.0)

        if ans == 1:
            sent = sents.get("true") or f"{qk} is true."
        else:
            sent = sents.get("false") or f"{qk} is false."

        if include_conf:
            try:
                sent = sent + conf_fmt.format(conf=float(conf))
            except Exception:
                pass

        lines.append(f"- {sent}")

    return "\n".join(lines)


Loaded VQA rows: 90


In [21]:
def _qa_get(qa: Dict[str, Any], key: str) -> Optional[Dict[str, Any]]:
    v = qa.get(key)
    return v if isinstance(v, dict) else None

def _qa_answer_is_one(qa: Dict[str, Any], key: str) -> bool:
    v = _qa_get(qa, key)
    if not v or "answer" not in v:
        return False
    try:
        return int(v["answer"]) == 1
    except Exception:
        return False

def _condition_met(cond: Dict[str, Any], qa: Dict[str, Any]) -> bool:
    # support {"all_true": [..]}
    req = cond.get("all_true")
    if isinstance(req, list):
        return all(_qa_answer_is_one(qa, k) for k in req)
    return True

def render_cues_block(meta_rec: Optional[Dict[str, Any]], schema: Dict[str, Any]) -> Optional[str]:
    # If no record or parse failed, return None (so prompt shows "(No cues available...)")
    if not meta_rec or not meta_rec.get("parse_ok", False):
        return None

    qa = meta_rec.get("qa") or {}
    rendering = schema.get("rendering", {})
    include_conf = bool(rendering.get("include_confidence", True))
    conf_fmt = rendering.get("confidence_format", " (confidence={conf:.2f})")
    prefix = rendering.get("prefix")  # optional; you can drop it to avoid duplication

    lines = []
    if prefix:
        lines.append(prefix)

    for q in schema.get("questions", []):
        qk = q.get("key")
        if not qk:
            continue

        sentences = q.get("sentences", {})
        cond = q.get("condition")

        # --- Conditional question whose condition is not met => render NA sentence (no confidence) ---
        if isinstance(cond, dict) and not _condition_met(cond, qa):
            na = sentences.get("note_condition_not_met") or sentences.get("na") or (
                f"{qk}: not applicable because prerequisite cues are not satisfied."
            )
            lines.append(f"- {na}")
            continue

        v = _qa_get(qa, qk)
        if not v or "answer" not in v:
            sent = sentences.get("unknown") or f"It is unclear whether {qk} is true in this frame."
            lines.append(f"- {sent}")
            continue

        ans = v.get("answer")
        conf = v.get("confidence", None)

        try:
            ans_int = int(ans)
        except Exception:
            ans_int = 0

        if ans_int == 1:
            sent = sentences.get("true") or f"{qk} is true."
        else:
            sent = sentences.get("false") or f"{qk} is false."

        # Only attach confidence if it's present AND you want it
        if include_conf and conf is not None:
            try:
                sent = sent + conf_fmt.format(conf=float(conf))
            except Exception:
                pass

        lines.append(f"- {sent}")

    return "\n".join(lines) if lines else None


## 3) Build C1 prompts + print first example

In [23]:
from typing import Optional, Literal

C1_MODE = Literal["image_only", "image_plus_cues", "cues_only"]

def build_c1_prompt(*, mode: C1_MODE, cues_block: Optional[str] = None) -> str:
    base = [
        "You are an expert surgical assistant evaluating Critical View of Safety (CVS).",
        "",
        "Task: Predict C1 ONLY.",
        "C1 (Two tubular structures): Are two AND ONLY two tubular structures clearly visible entering the gallbladder "
        "(i.e., consistent with cystic duct and cystic artery), with no additional tubular structure entering the gallbladder?",
        "",
        "Decision rule for c1:",
        "- Set c1 = 1 ONLY if you can clearly identify exactly TWO tubular structures entering the gallbladder.",
        "- Otherwise set c1 = 0 (including: unclear visibility, ambiguity, partial occlusion, too much fat/fibrous tissue, or >2 candidates).",
        "",
    ]

    if mode in ("image_plus_cues", "cues_only"):
        base += [
            "Automated visual cues extracted from this frame (may be noisy):",
            cues_block if cues_block else "(No cues available for this frame.)",
            "",
        ]

        if mode == "image_plus_cues":
            base += [
                "How to use these cues:",
                "- Use these cues to guide attention, but do NOT let them override clear visual evidence in the image.",
                "- If the cues conflict with what you see, trust the image and briefly mention the conflict in the rationale.",
                "",
            ]
        else:  # cues_only
            base += [
                "Important (cues-only setting):",
                "- You will NOT be shown the image.",
                "- Base your decision ONLY on the cues above.",
                "- If the cues are insufficient to decide, set c1 = 0 and use score≈0.5 with a rationale explaining ambiguity.",
                "",
            ]

    base += [
        "Return ONLY valid JSON (no markdown, no extra keys):",
        '{"rationale": string, "c1": 0 or 1, "score": float between 0 and 1}',
        "",
        "Score semantics (belief that C1=1):",
        "- score = 1.0 : very confident C1 = 1",
        "- score = 0.6 : weak evidence leaning C1 = 1",
        "- score = 0.5 : truly ambiguous / cannot decide",
        "- score = 0.4 : weak evidence leaning C1 = 0",
        "- score = 0.0 : very confident C1 = 0",
        "",
        "Consistency requirement:",
        "- If c1 = 1, score must be >= 0.5.",
        "- If c1 = 0, score must be <= 0.5.",
    ]
    return "\n".join(base)


# Load schema (you probably already have this)
# schema = json.loads(SCHEMA_PATH.read_text())

manifest_rows = list(iter_jsonl(MANIFEST_JSONL))

cues_block = None
for rec in manifest_rows:
    vid = rec.get("id") or rec.get("video_id")
    sp = rec.get("split", "train")
    fids = rec.get("frame_ids") or []
    if vid is None or not fids:
        continue

    for fid in fids:
        try:
            fid_int = int(fid)
        except Exception:
            continue

        vqa_rec = vqa_idx.get(kf(str(vid), str(sp), fid_int))
        cues_block = render_cues_block(vqa_rec, schema)

        if cues_block is not None:
            print("Example cues_block:\n", cues_block)
            break
    if cues_block is not None:
        break


SYSTEM_PROMPT_C1 = (
    "You are a careful surgical vision assistant. "
    "If an image is provided, use it as primary evidence. "
    "If only automated visual cues are provided, base your answer ONLY on those cues. "
    "Output must be valid JSON matching the schema exactly."
)

prompt_img_only  = build_c1_prompt(mode="image_only", cues_block=None)
prompt_img_plus  = build_c1_prompt(mode="image_plus_cues", cues_block=cues_block)
prompt_cues_only = build_c1_prompt(mode="cues_only", cues_block=cues_block)


print("\n--- PROMPT (IMAGE ONLY) ---\n")
print(prompt_img_only)

print("\n--- PROMPT (IMAGE + CUES) ---\n")
print(prompt_img_plus)

print("\n--- PROMPT (CUES ONLY, NO IMAGE) ---\n")
print(prompt_cues_only)

Example cues_block:
 Automated VQA cues (may be noisy):
- The cystic duct is clearly visible in this frame. (confidence=0.80)
- The cystic artery is not clearly visible in this frame. (confidence=0.70)
- Both the cystic duct and the cystic artery are not clearly visible in this frame. (confidence=0.70)
- Separation between the cystic duct and cystic artery cannot be assessed because both structures are not clearly visible.

--- PROMPT (IMAGE ONLY) ---

You are an expert surgical assistant evaluating Critical View of Safety (CVS).

Task: Predict C1 ONLY.
C1 (Two tubular structures): Are two AND ONLY two tubular structures clearly visible entering the gallbladder (i.e., consistent with cystic duct and cystic artery), with no additional tubular structure entering the gallbladder?

Decision rule for c1:
- Set c1 = 1 ONLY if you can clearly identify exactly TWO tubular structures entering the gallbladder.
- Otherwise set c1 = 0 (including: unclear visibility, ambiguity, partial occlusion, t

## 4) Run C1 prediction for all frames → outputs

In [24]:
def normalize_c1(obj: Dict[str, Any]) -> Tuple[bool, List[str], Dict[str, Any]]:
    errs: List[str] = []
    if not isinstance(obj, dict):
        return False, ["not_a_dict"], {"rationale": "", "c1": 0, "score": 0.5}

    rationale = obj.get("rationale", "")
    c1 = obj.get("c1", 0)
    score = obj.get("score", 0.5)

    try:
        c1_int = int(c1)
        if c1_int not in (0, 1):
            errs.append("c1_not_0_1")
            c1_int = 0
    except Exception:
        errs.append("c1_parse_fail")
        c1_int = 0

    try:
        score_f = float(score)
        score_f = max(0.0, min(1.0, score_f))
    except Exception:
        errs.append("score_parse_fail")
        score_f = 0.5

    if c1_int == 1 and score_f < 0.5:
        errs.append("score_inconsistent_with_c1_1")
        score_f = 0.5
    if c1_int == 0 and score_f > 0.5:
        errs.append("score_inconsistent_with_c1_0")
        score_f = 0.5

    return (len(errs) == 0), errs, {"rationale": str(rationale), "c1": c1_int, "score": score_f}


# --- NEW: third output, without renaming existing ones ---
OUT_CUES_ONLY = (REPO_ROOT / "outputs" / "c1_pred" / f"c1__cues_only__{MANIFEST_JSONL.stem}.jsonl").resolve()

REGENERATE_C1 = True
if REGENERATE_C1:
    for p in (OUT_WITH_META, OUT_NO_META, OUT_CUES_ONLY):
        if p.exists():
            p.unlink()
            print("Deleted:", p)

start_time = time.time()
n_frames = n_missing = n_meta_used = 0
manifest_rows = list(iter_jsonl(MANIFEST_JSONL))

for rec in tqdm(manifest_rows, desc="C1 videos"):
    video_id = rec.get("id") or rec.get("video_id")
    split = rec.get("split", "train")
    frame_ids = rec.get("frame_ids") or []
    if video_id is None or not frame_ids:
        continue

    for fid in tqdm(frame_ids, desc=f"frames {str(video_id)[:8]}", leave=False):
        try:
            fid_int = int(fid)
        except Exception:
            continue

        img_p = frame_path(str(split), str(video_id), fid_int)
        abs_img = abs_posix(img_p)

        # ---- load cues (from VQA jsonl index) ----
        meta_rec = vqa_idx.get(kf(str(video_id), str(split), fid_int))
        meta_block = render_metadata_sentences(meta_rec, schema)  # keep your existing function name
        cues_present = meta_block is not None
        if cues_present:
            n_meta_used += 1

        # ---- missing image: still write stubs for all 3 conditions ----
        if not img_p.exists():
            n_missing += 1
            stub = {
                "task": "c1_prediction",
                "video_id": str(video_id),
                "split": str(split),
                "frame_id": fid_int,
                "image_path": abs_img,
                "model": MODEL_ID,
                "raw_response": "",
                "parse_ok": False,
                "parse_errors": ["missing_image"],
                "pred": {"rationale": "", "c1": 0, "score": 0.5},
            }
            append_jsonl(OUT_WITH_META, dict(stub, condition="with_metadata", metadata_present=cues_present, metadata_source_jsonl=abs_posix(VQA_OUT_JSONL)))
            append_jsonl(OUT_NO_META,   dict(stub, condition="no_metadata"))
            append_jsonl(OUT_CUES_ONLY, dict(stub, condition="cues_only", metadata_present=cues_present, metadata_source_jsonl=abs_posix(VQA_OUT_JSONL)))
            continue

        img = load_image(img_p)
        n_frames += 1

        # ============================================================
        # A) IMAGE + CUES  (keep saving to OUT_WITH_META)
        # ============================================================
        prompt_with = build_c1_prompt(mode="image_plus_cues", cues_block=meta_block)
        t0 = time.time()
        resp_with = model([(prompt_with, img)], system_prompt=SYSTEM_PROMPT_C1)[0]
        dt_with = time.time() - t0
        try:
            obj_with = extract_json_obj(resp_with)
            ok_with, errs_with, pred_with = normalize_c1(obj_with)
        except Exception as e:
            ok_with, errs_with, pred_with = False, [f"parse_failed: {type(e).__name__}: {e}"], {"rationale": "", "c1": 0, "score": 0.5}

        append_jsonl(OUT_WITH_META, {
            "task": "c1_prediction",
            "condition": "with_metadata",  # keep your existing label to avoid renaming everywhere
            "video_id": str(video_id),
            "split": str(split),
            "frame_id": fid_int,
            "image_path": abs_img,
            "model": MODEL_ID,
            "latency_sec": dt_with,
            "metadata_present": cues_present,
            "metadata_source_jsonl": abs_posix(VQA_OUT_JSONL),
            "raw_response": resp_with,
            "parse_ok": ok_with,
            "parse_errors": errs_with,
            "pred": pred_with,
        })

        # ============================================================
        # B) IMAGE ONLY  (keep saving to OUT_NO_META)
        # ============================================================
        prompt_no = build_c1_prompt(mode="image_only", cues_block=None)
        t0 = time.time()
        resp_no = model([(prompt_no, img)], system_prompt=SYSTEM_PROMPT_C1)[0]
        dt_no = time.time() - t0
        try:
            obj_no = extract_json_obj(resp_no)
            ok_no, errs_no, pred_no = normalize_c1(obj_no)
        except Exception as e:
            ok_no, errs_no, pred_no = False, [f"parse_failed: {type(e).__name__}: {e}"], {"rationale": "", "c1": 0, "score": 0.5}

        append_jsonl(OUT_NO_META, {
            "task": "c1_prediction",
            "condition": "no_metadata",
            "video_id": str(video_id),
            "split": str(split),
            "frame_id": fid_int,
            "image_path": abs_img,
            "model": MODEL_ID,
            "latency_sec": dt_no,
            "raw_response": resp_no,
            "parse_ok": ok_no,
            "parse_errors": errs_no,
            "pred": pred_no,
        })

        # ============================================================
        # C) CUES ONLY (NO IMAGE)  -> write to OUT_CUES_ONLY
        # ============================================================
        prompt_cues_only = build_c1_prompt(mode="cues_only", cues_block=meta_block)

        t0 = time.time()
        # text-only call: try the simplest form first
        try:
            resp_cues_only = model([prompt_cues_only], system_prompt=SYSTEM_PROMPT_C1)[0]
        except Exception:
            # fallback: in case your wrapper expects a list of "messages"
            resp_cues_only = model([(prompt_cues_only,)], system_prompt=SYSTEM_PROMPT_C1)[0]
        dt_cues_only = time.time() - t0

        try:
            obj_c = extract_json_obj(resp_cues_only)
            ok_c, errs_c, pred_c = normalize_c1(obj_c)
        except Exception as e:
            ok_c, errs_c, pred_c = False, [f"parse_failed: {type(e).__name__}: {e}"], {"rationale": "", "c1": 0, "score": 0.5}

        append_jsonl(OUT_CUES_ONLY, {
            "task": "c1_prediction",
            "condition": "cues_only",
            "video_id": str(video_id),
            "split": str(split),
            "frame_id": fid_int,
            "image_path": abs_img,  # keep for traceability, but not shown to model
            "model": MODEL_ID,
            "latency_sec": dt_cues_only,
            "metadata_present": cues_present,
            "metadata_source_jsonl": abs_posix(VQA_OUT_JSONL),
            "raw_response": resp_cues_only,
            "parse_ok": ok_c,
            "parse_errors": errs_c,
            "pred": pred_c,
        })

elapsed = time.time() - start_time
print(f"Done C1. frames={n_frames}, missing_img={n_missing}, cues_available={n_meta_used}")
print(f"Wrote:\n  - {OUT_WITH_META}\n  - {OUT_NO_META}\n  - {OUT_CUES_ONLY}")
print(f"Elapsed: {elapsed/60:.1f} min")


Deleted: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__with_meta__train_dev5.jsonl
Deleted: /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__no_meta__train_dev5.jsonl


C1 videos:   0%|          | 0/5 [00:00<?, ?it/s]

frames 925421af:   0%|          | 0/18 [00:00<?, ?it/s]

frames b5e98834:   0%|          | 0/18 [00:00<?, ?it/s]

frames 534c4a9e:   0%|          | 0/18 [00:00<?, ?it/s]

frames 3cc65e5e:   0%|          | 0/18 [00:00<?, ?it/s]

frames e6093fb5:   0%|          | 0/18 [00:00<?, ?it/s]

Done C1. frames=90, missing_img=0, cues_available=90
Wrote:
  - /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__with_meta__train_dev5.jsonl
  - /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__no_meta__train_dev5.jsonl
  - /shared_data0/weiqiuy/lvlm_cvs_reasoning/outputs/c1_pred/c1__cues_only__train_dev5.jsonl
Elapsed: 11.7 min


## 5) Metrics (accuracy, TP/FP/TN/FN, AP, mAP)

In [28]:
from sklearn.metrics import average_precision_score
import math

def load_frame_gt(video_id: str, split: str) -> Dict[int, int]:
    p = (GT_LABELS_ROOT / split / "labels" / video_id / "frame.csv").resolve()
    if not p.exists():
        return {}
    out: Dict[int, int] = {}
    with p.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fid = int(row["frame_id"])
            votes = [int(row.get("c1_rater1", 0)), int(row.get("c1_rater2", 0)), int(row.get("c1_rater3", 0))]
            if VOTE_RULE == "any":
                out[fid] = 1 if any(v == 1 for v in votes) else 0
            else:
                out[fid] = 1 if sum(votes) >= 2 else 0
    return out

# Build GT index for manifest videos
gt_idx: Dict[str, int] = {}  # key -> label
manifest_rows = list(iter_jsonl(MANIFEST_JSONL))
for rec in manifest_rows:
    vid = rec.get("id") or rec.get("video_id")
    sp = rec.get("split", "train")
    if vid is None:
        continue
    per = load_frame_gt(str(vid), str(sp))
    for fid, y in per.items():
        gt_idx[kf(str(vid), str(sp), int(fid))] = int(y)
print("GT frames indexed:", len(gt_idx))

def load_preds(path: Path) -> List[Dict[str, Any]]:
    return list(iter_jsonl(path)) if path.exists() else []

with_meta_rows = load_preds(OUT_WITH_META)
no_meta_rows = load_preds(OUT_NO_META)
print("Pred rows:", len(with_meta_rows), "(with meta),", len(no_meta_rows), "(no meta)")

# Optional third condition if you created it
try:
    cues_only_rows = load_preds(OUT_CUES_ONLY)  # will work if defined
    print("Pred rows:", len(cues_only_rows), "(cues only)")
except Exception:
    cues_only_rows = None

def confusion(y_true: List[int], y_pred: List[int]) -> Dict[str, int]:
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn}

def _safe_float(x: Any, default: float) -> float:
    try:
        v = float(x)
        if math.isnan(v) or math.isinf(v):
            return default
        return v
    except Exception:
        return default

def eval_rows(
    rows: List[Dict[str, Any]],
    score_mode: str,
    *,
    drop_parse_fail: bool = True,
    print_parse_failures: bool = True,
    max_print: int = 25,
) -> Dict[str, Any]:
    """
    score_mode:
      - "binary": use pred.c1 as score (0/1)
      - "continuous": use pred.score as score (0..1)

    Defaults:
      - missing/invalid c1 -> 0
      - missing/invalid score -> 0.5
    """
    y_true: List[int] = []
    y_pred: List[int] = []
    y_score: List[float] = []
    vids: List[str] = []

    n_total = 0
    n_missing_gt = 0
    n_parse_fail = 0
    n_printed = 0

    def _short(s: Any, n: int = 260) -> str:
        s = "" if s is None else str(s)
        s = s.replace("\n", "\\n")
        return (s[:n] + "…") if len(s) > n else s

    for r in rows:
        n_total += 1

        parse_ok = bool(r.get("parse_ok", False))
        if drop_parse_fail and not parse_ok:
            n_parse_fail += 1
            if print_parse_failures and n_printed < max_print:
                vid = r.get("video_id")
                sp = r.get("split")
                fid = r.get("frame_id")
                errs = r.get("parse_errors", [])
                raw = r.get("raw_response", "")
                print("\n[PARSE_FAIL] ",
                      f"video_id={vid} split={sp} frame_id={fid} "
                      f"errors={errs} raw_response_snippet='{_short(raw)}'")
                n_printed += 1
            continue

        vid = r.get("video_id")
        sp = r.get("split")
        fid = r.get("frame_id")
        if vid is None or sp is None or fid is None:
            continue

        kk = kf(str(vid), str(sp), int(fid))
        if kk not in gt_idx:
            n_missing_gt += 1
            continue

        pred = r.get("pred") or {}
        yt = int(gt_idx[kk])

        # predicted class (default 0)
        yp = int(_safe_float(pred.get("c1", 0), 0.0))
        yp = 1 if yp == 1 else 0

        # score for AP (default 0.5 for continuous)
        if score_mode == "binary":
            sc = float(yp)
        else:
            sc = _safe_float(pred.get("score", 0.5), 0.5)
            sc = max(0.0, min(1.0, sc))

        y_true.append(yt)
        y_pred.append(yp)
        y_score.append(sc)
        vids.append(str(vid))

    acc = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp) / len(y_true) if y_true else 0.0
    conf = confusion(y_true, y_pred)

    ap_global = average_precision_score(y_true, y_score) if len(set(y_true)) > 1 else float("nan")

    return {
        "n_total_rows": n_total,
        "n_used": len(y_true),
        "n_parse_fail_dropped": n_parse_fail if drop_parse_fail else 0,
        "n_missing_gt": n_missing_gt,
        "accuracy": acc,
        **conf,
        "AP_global": ap_global,
    }


def report(name: str, rows: List[Dict[str, Any]]) -> None:
    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)
    print("\n[Binary score = pred.c1 (0/1)]")
    print(eval_rows(rows, "binary", drop_parse_fail=True))
    print("\n[Continuous score = pred.score (0..1)]")
    print(eval_rows(rows, "continuous", drop_parse_fail=True))

report("WITH METADATA (image + cues)", with_meta_rows)
report("NO METADATA (image only)", no_meta_rows)
if cues_only_rows is not None:
    report("CUES ONLY (no image)", cues_only_rows)


GT frames indexed: 90
Pred rows: 90 (with meta), 90 (no meta)
Pred rows: 90 (cues only)

WITH METADATA (image + cues)

[Binary score = pred.c1 (0/1)]

[PARSE_FAIL]  video_id=534c4a9e-5b08-45e7-9d34-b24b0c6291fe split=train frame_id=1500 errors=["parse_failed: JSONDecodeError: Expecting ',' delimiter: line 1 column 405 (char 404)"] raw_response_snippet='{"rationale":"The image shows exactly two tubular structures entering the gallbladder, which appear to be distinct and clearly visible. There is no evidence of additional tubular structures or obstructions such as fat or fibrous tissue that obscure visibility.…'
{'n_total_rows': 90, 'n_used': 89, 'n_parse_fail_dropped': 1, 'n_missing_gt': 0, 'accuracy': 0.7303370786516854, 'tp': 7, 'fp': 10, 'tn': 58, 'fn': 14, 'AP_global': 0.2945582727473012}

[Continuous score = pred.score (0..1)]

[PARSE_FAIL]  video_id=534c4a9e-5b08-45e7-9d34-b24b0c6291fe split=train frame_id=1500 errors=["parse_failed: JSONDecodeError: Expecting ',' delimiter: line 

1. Reduced fp (although also tp).